# Block 1 — Structural geology with `apsg`

*Modern Workflow in Geology, 8 October 2026 — IPSG, Charles University*

`apsg` (*A Python package for structural geologists*) represents orientation data as
**objects**: a lineation knows it is a lineation, a set of foliations knows how to
compute its own mean or eigenvectors, and a stereonet knows how to draw any of them.

| Section | Topic |
|---|---|
| 1.1 | Vectors |
| 1.2 | Lineations and foliations |
| 1.3 | Pairs and faults |
| 1.4 | Feature sets |
| 1.5 | Stereonets |
| 1.6 | Population analysis: statistics and eigenvectors |
| 1.7 | Fold axis from folded foliations |
| 1.8 | Reading field data with pandas |
| 1.9 | Saving vector graphics |
| 1.10 | Hands-on exercise |

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from apsg import *                     # vec, lin, fol, pair, fault, sets, StereoNet, ...
from apsg.pandas import pd, gbf        # pandas with apsg accessors registered

DATA = Path("data")
OUT = Path("output")
OUT.mkdir(exist_ok=True)
np.random.seed(2026)                   # reproducible random examples

## 1.1 Vectors

`vec` is a 3D vector in a geographic coordinate system (x = north, y = east, z = down).
It can be created from components or from azimuth and inclination.

In [ ]:
u = vec(1, -2, 3)
v = vec(120, 60)          # azimuth 120, inclination 60
u, v

In [ ]:
print("length      :", abs(u))
print("sum         :", u + v)
print("dot product :", u @ v)
print("cross prod. :", u.cross(v), u ** v)
print("angle (deg) :", u.angle(v))
print("u rotated 45° around v:", u.rotate(v, 45))

## 1.2 Lineations and foliations

`lin(azi, inc)` is a lineation (trend/plunge), `fol(azi, inc)` a planar feature given by
**dip direction / dip**. Both are *axial* — `lin(120, 60)` and its opposite are the same
line — and always refer to the lower hemisphere.

In [ ]:
l = lin(120, 60)
f = fol(216, 62)
l, f

Other notations are accepted on input and can be switched on output:

In [ ]:
print(fol("N30E,40NW"), lin("N45E,30"))            # quadrant notation
with apsg_conf_context(notation="rhr"):             # right-hand-rule strike/dip
    print(f)

Conversions are explicit — a foliation stores its **pole**, so turning a plane into a
vector gives the pole, and turning a vector into a plane gives the plane normal to it:

In [ ]:
print("pole to f          :", lin(f))
print("plane normal to l  :", fol(l))
print("dip vector of f    :", lin(f.dipvec()))
print("line with rake 30° :", lin(f.rake(30)))

The `**` operator is the geological cross product: two planes give their **intersection
lineation**, two lines give the **plane containing both**.

In [ ]:
f1, f2 = fol(250, 40), fol(160, 30)
print("intersection of two planes :", f1 ** f2)
print("plane through two lines    :", lin(250, 40) ** lin(160, 30))
print("angle between the planes   :", f1.angle(f2))

## 1.3 Pairs and faults

A `pair` is a plane with a line lying in it (e.g. foliation + stretching lineation).
Measured pairs never fit perfectly; apsg adjusts them and reports the misfit.

In [ ]:
p = pair(120, 40, 162, 28)       # fol azi, inc, lin azi, inc
print(p, "misfit:", round(p.misfit, 2), "deg")
print("plane:", p.fol, " line:", p.lin, " rake:", round(p.rake, 1))

A `fault` is a pair with a **sense of movement**: `1`/`"n"` normal, `-1`/`"r"` reverse,
`"s"` sinistral, `"d"` dextral. From it apsg derives the kinematic **P**, **T** and
**M** (movement-plane) axes used in Block 2.

In [ ]:
ft = fault(120, 60, 110, 58, "n")
print(ft)
print("P:", ft.p, " T:", ft.t, " M-plane:", ft.m)

## 1.4 Feature sets

Collections of the same feature type: `vecset`, `linset`, `folset`, `pairset`,
`faultset`. `G([...])` builds the right one automatically. Sets behave like lists (index,
slice, `len`) and add group-level methods.

In [ ]:
L = G([lin(120, 60), lin(116, 50), lin(132, 45), lin(90, 60), lin(84, 52)], name="L1")
L, len(L), L[0]

For demonstrations we can draw random samples from a Fisher distribution around a mean
orientation (larger `kappa` = tighter cluster):

In [ ]:
cluster = linset.random_fisher(position=lin(120, 40), kappa=30, n=100, name="L cluster")
cluster

## 1.5 Stereonets

`StereoNet()` creates an equal-area, lower-hemisphere net. Add layers, then `.show()`.

| Method | Draws |
|---|---|
| `s.line(l)` / `s.pole(f)` | points (lineations, poles) |
| `s.great_circle(f)` | planes |
| `s.contour(data)` | density contours (Kamb by default) |
| `s.confidence(data)` | confidence cone around the mean |
| `s.pair(p)`, `s.fault(f)`, `s.cone(c)` | composite features |

In [ ]:
s = StereoNet(title="Planes, poles and a line")
s.great_circle(f1, f2, label="planes")
s.pole(f1, f2, color="k", label="poles")
s.line(f1 ** f2, color="r", ms=8, label="intersection")
s.show()

In [ ]:
s = StereoNet(title="Density contours")
s.contour(cluster, levels=8, colorbar=True)
s.line(cluster, color="k", ms=3)
s.confidence(cluster, method="fisher", level=0.95, color="r")
s.show()

`StereoNet(kind="equal-angle")` gives a Wulff net, `hemisphere="upper"` the upper
hemisphere, `grid=False` removes the grid.

## 1.6 Population analysis: statistics and eigenvectors

**Vector (Fisher) statistics** — for clustered data: resultant (mean) direction,
precision `k`, 95 % confidence cone `alpha`.

In [ ]:
print("mean direction :", cluster.R())
stats = cluster.fisher_statistics()
print(f"Fisher k = {stats['k']:.1f},  alpha95 = {stats['alpha']:.1f} deg")
print("spherical variance:", round(cluster.var(), 3))

**Orientation tensor** — works for both clusters and girdles. Its eigenvectors are the
principal axes; the eigenvalues describe the fabric shape (Vollmer's *P*, *G*, *R*
indices: point, girdle, random).

In [ ]:
# poles spread along a great circle normal to the axis 300/20
rng = np.random.default_rng(1)
girdle = G([fol(lin(30, 0).rotate(lin(300, 20), a)) for a in rng.normal(0, 60, 150)])

for name, data in [("cluster", cluster), ("girdle", girdle)]:
    ot = data.ortensor()
    print(f"{name:8s} E1,E2,E3 = {np.round(ot.eigenvalues(), 3)}   P={ot.P:.2f} G={ot.G:.2f} R={ot.R:.2f}")

In [ ]:
ot = girdle.ortensor()
s = StereoNet(title="Girdle of poles")
s.pole(girdle, ms=3)
s.line(*ot.eigenlins(), color="r", ms=8)             # principal axes
s.great_circle(ot.eigenfols(2), color="r")           # best-fit girdle plane
s.show()

## 1.7 Fold axis from folded foliations

For a cylindrical fold the poles of the folded surface lie on a great circle (a
**π-girdle**); its pole — the eigenvector with the *smallest* eigenvalue — is the fold
axis (**β**). Here we use real data from `data/structures.csv`: S1 is folded by D3, and
L3 was measured independently in the field.

In [ ]:
df = pd.read_csv(DATA / "structures.csv")
df.head()

In [ ]:
dg = df.groupby("structure")
dg.groups.keys()

In [ ]:
dg.get_group('S1')

In [ ]:
S1 = dg.get_group('S1').fol(name="S1")      # pandas accessor fol -> FoliationSet
L3 = dg.get_group('L3').lin(name="L3")      # pandas accessor lin -> LineationSet
beta = S1.ortensor().eigenlins(2)           # smallest eigenvector = girdle axis
print("beta (S1 girdle axis):", beta)
print("mean L3              :", L3.ortensor().eigenlins(0))
print("angle between them   :", round(beta.angle(L3.ortensor().eigenlins(0)), 1), "deg")

In [ ]:
s = StereoNet(title="D3 folding of S1")
s.contour(S1, levels=6, cmap="Greys", alpha=0.6)
s.pole(S1, ms=3, label="S1 poles")
s.great_circle(S1.ortensor().eigenfols(2), color="k", label="pi-girdle")
s.line(L3, color="g", ms=3, label="L3")
s.line(beta, color="r", ms=10, label="beta axis")
s.show()

## 1.8 Reading field data with pandas

`from apsg.pandas import pd` registers accessors on DataFrames: `df.lin`, `df.fol`,
`df.vec`, `df.fault`. By default they read the columns `azi` and `inc`; `.set_columns()`
points them elsewhere. `gbf` holds ready-made functions for `groupby`.

In [ ]:
# mean L3 per site
dg.get_group("L3").lin.groupby("site").apply(gbf.mean).head()

In [ ]:
# a quick stereonet straight from the DataFrame
dg.get_group("S1").fol.plot(kind="pole", label="S1")

In [ ]:
# mean S1 per site
s1s = dg.get_group("S1").fol.groupby("site").apply(gbf.mean)
s = StereoNet(title="S1 averaged by site")
s.pole(folset(s1s))
s.show()

Many field notebooks are stored **wide** — one column pair per structure. Point the
accessor at the right columns:

In [ ]:
wide = pd.read_csv(DATA / "structures_wide.csv")
wide.replace(np.nan, "")

In [ ]:
wide = pd.read_csv(DATA / "structures_wide.csv")
wide.fol.set_columns(azi="S1_azi", inc="S1_inc")
S1_wide = wide.fol(name="S1")        # rows with missing values are skipped
S1_wide

## 1.9 Saving vector graphics

`StereoNet.savefig()` passes everything to matplotlib: the file extension picks the format
(`.pdf`, `.svg`, `.eps` are vector — editable in Inkscape/Illustrator; `.png` needs
`dpi=`).

In [ ]:
s = StereoNet(title="S1 and L3")
s.great_circle(S1, color="0.7", lw=0.5)
s.line(L3, color="g", ms=4, label="L3")
s.line(beta, color="r", ms=10, label="beta")
s.savefig(OUT / "S1_L3.pdf")
s.savefig(OUT / "S1_L3.svg")
s.savefig(OUT / "S1_L3.png", dpi=300)
sorted(p.name for p in OUT.glob("S1_L3*"))

## 1.10 Hands-on exercise

`data/field_measurements.csv` holds a small field survey of a folded sequence: bedding
(`S0`), cleavage (`S1`) and intersection lineation (`L1`) at 12 stations, in dip
direction / dip.

1. Load the file and create a `FoliationSet` for S0 and S1 and a `LineationSet` for L1.
2. Plot S0 poles with density contours, S1 as great circles and L1 as points.
3. Find the fold axis (β) from the S0 poles. Compare it with the mean L1. Does β lie
   in the mean S1 plane (axial-planar cleavage)?
4. Compute the mean S1 and its Fisher α95.
5. Save the stereonet as PDF and SVG to `output/`.

Write your code in the empty cells; a worked solution follows.

In [ ]:
# your code here

In [ ]:
# your code here

<details>
<summary><b>Click here to see the solution</b></summary>

```python
field = pd.read_csv(DATA / "field_measurements.csv")
S0 = field[field.structure == "S0"].fol(name="S0")
S1 = field[field.structure == "S1"].fol(name="S1")
L1 = field[field.structure == "L1"].lin(name="L1")

beta = S0.ortensor().eigenlins(2)
S1_mean = S1.R()
print("beta axis        :", beta)
print("mean L1          :", L1.R())
print("mean S1          :", S1_mean, f"alpha95 = {S1.fisher_statistics()['alpha']:.1f} deg")
print("beta to S1 plane :", round(90 - beta.angle(lin(S1_mean)), 1), "deg (0 = beta lies in S1)")

s = StereoNet(title="Field survey")
s.contour(S0, levels=6, cmap="Blues", alpha=0.7)
s.pole(S0, color="b", ms=3, label="S0 poles")
s.great_circle(S0.ortensor().eigenfols(2), color="b", ls="--", label="pi-girdle S0")
s.great_circle(S1_mean, color="r", label="mean S1")
s.confidence(S1, method="fisher", color="r")
s.line(L1, color="g", ms=4, label="L1")
s.line(beta, color="k", ms=10, label="beta")
s.savefig(OUT / "field_survey.pdf")
s.savefig(OUT / "field_survey.svg")
s.show()
```

</details>

## Recap

| Task | Code |
|---|---|
| Create features | `vec(1,2,3)`, `lin(120,60)`, `fol(216,62)`, `pair(...)`, `fault(..., "n")` |
| Convert | `lin(f)` (pole), `fol(l)` (normal plane), `f1 ** f2` (intersection) |
| Collections | `G([...])`, `linset`, `folset`, `df.fol()`, `df.lin()` |
| Plot | `StereoNet()` → `.line/.pole/.great_circle/.contour/.confidence` → `.show()/.savefig()` |
| Statistics | `.R()`, `.fisher_statistics()`, `.var()`, `.ortensor().eigenlins()/.eigenfols()` |
| Fold axis | `S.ortensor().eigenlins(2)` |

Next: **Block 2** — faults, tensors and stress inversion (`02_apsg_advanced.ipynb`).